In [2]:
import gradio as gr
from __future__ import annotations
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from io import BytesIO
from PIL import Image

# ----------------------------
# 1) World model (Blocks World)
# ----------------------------

@dataclass(frozen=True)
class Obj:
    name: str
    color: str
    shape: str  # "cube", "pyramid", "block" (generic)

@dataclass
class World:
    objects: Dict[str, Obj]
    on: Dict[str, Optional[str]]  # on[x] = y means x is on y; None means on table
    holding: Optional[str] = None

    def is_clear(self, obj_name: str) -> bool:
        """True if nothing is on top of obj_name."""
        return all(support != obj_name for support in self.on.values())

    def top_of(self, obj_name: str) -> Optional[str]:
        """Return object that is on top of obj_name (if any)."""
        for x, support in self.on.items():
            if support == obj_name:
                return x
        return None

    def describe(self) -> str:
        lines = []
        for x in sorted(self.objects):
            support = self.on.get(x, None)
            if support is None:
                lines.append(f"{x} is on the table")
            else:
                lines.append(f"{x} is on {support}")
        lines.append(f"Holding: {self.holding}")
        return "\n".join(lines)

    def pickup(self, x: str) -> None:
        if self.holding is not None:
            raise RuntimeError("Already holding something.")
        if not self.is_clear(x):
            raise RuntimeError(f"Cannot pick up {x}: not clear.")
        self.holding = x
        self.on[x] = None

    def put_on(self, x: str, y: str) -> None:
        if self.holding != x:
            raise RuntimeError(f"Not holding {x}.")
        if not self.is_clear(y):
            raise RuntimeError(f"Cannot place on {y}: {y} not clear.")
        self.on[x] = y
        self.holding = None


# ----------------------------------------
# 2) Reference grounding (simple semantics)
# ----------------------------------------

def resolve_ref(world: World, color: Optional[str], shape: Optional[str]) -> List[str]:
    """Return object names matching the requested properties."""
    matches = []
    for name, obj in world.objects.items():
        if color is not None and obj.color != color:
            continue
        if shape is not None:
            if shape != "block" and obj.shape != shape:
                continue
        matches.append(name)
    return matches


# ----------------------------------------
# 3) Planning / inference (toy planner)
# ----------------------------------------

def plan_pickup(world: World, x: str) -> List[Tuple[str, str, Optional[str]]]:
    """Plan to pick up x. If x is not clear, move blockers to table first."""
    plan = []
    blocker = world.top_of(x)
    if blocker is not None:
        plan += plan_pickup(world, blocker)
        plan.append(("put_on", blocker, None))
    plan.append(("pickup", x, None))
    return plan


def plan_put(world: World, x: str, y: str) -> List[Tuple[str, str, Optional[str]]]:
    """Plan to put x on y, ensuring both are feasible."""
    plan = []
    blocker = world.top_of(y)
    if blocker is not None:
        plan += plan_pickup(world, blocker)
        plan.append(("put_on", blocker, None))
    plan += plan_pickup(world, x)
    plan.append(("put_on", x, y))
    return plan


def execute_plan(world: World, plan: List[Tuple[str, str, Optional[str]]]) -> None:
    for action, obj, target in plan:
        if action == "pickup":
            world.pickup(obj)
        elif action == "put_on":
            if target is None:
                if world.holding != obj:
                    raise RuntimeError("Planner/executor mismatch.")
                world.on[obj] = None
                world.holding = None
            else:
                world.put_on(obj, target)
        else:
            raise ValueError(f"Unknown action: {action}")


# ----------------------------------------
# 4) Parsing
# ----------------------------------------

COLORS = {"red", "green", "blue", "yellow"}
SHAPES = {"cube", "pyramid", "block"}

def parse_command(text: str) -> dict:
    """Minimal rule-based parser for demo commands."""
    t = text.lower().replace("?", "").strip()
    tokens = t.split()

    if t.startswith("pick up"):
        color = next((w for w in tokens if w in COLORS), None)
        shape = next((w for w in tokens if w in SHAPES), "block")
        return {"intent": "PICKUP", "ref": {"color": color, "shape": shape}}

    if t.startswith("put"):
        if "on" not in tokens:
            raise ValueError("Expected 'on' in put command.")
        on_i = tokens.index("on")
        left = tokens[1:on_i]
        right = tokens[on_i+1:]

        color_x = next((w for w in left if w in COLORS), None)
        shape_x = next((w for w in left if w in SHAPES), "block")
        color_y = next((w for w in right if w in COLORS), None)
        shape_y = next((w for w in right if w in SHAPES), "block")

        return {
            "intent": "PUT_ON",
            "x": {"color": color_x, "shape": shape_x},
            "y": {"color": color_y, "shape": shape_y},
        }

    raise ValueError("Unknown command format.")


# ----------------------------------------
# 5) Dialogue manager
# ----------------------------------------

def choose_unique(matches: List[str], what: str) -> str:
    if not matches:
        raise ValueError(f"I can't find any {what} that matches your description.")
    if len(matches) > 1:
        raise ValueError(f"I don't understand which {what} you mean: {matches}")
    return matches[0]


def interpret_and_act(world: World, utterance: str) -> str:
    """Execute command and return response message."""
    parsed = parse_command(utterance)

    if parsed["intent"] == "PICKUP":
        ref = parsed["ref"]
        matches = resolve_ref(world, ref["color"], ref["shape"])
        x = choose_unique(matches, f"{ref['color'] or ''} {ref['shape']}".strip())
        plan = plan_pickup(world, x)
        execute_plan(world, plan)
        return f"OK. Picked up {x}. Plan: {plan}"

    elif parsed["intent"] == "PUT_ON":
        mx = resolve_ref(world, parsed["x"]["color"], parsed["x"]["shape"])
        my = resolve_ref(world, parsed["y"]["color"], parsed["y"]["shape"])
        x = choose_unique(mx, f"{parsed['x']['color'] or ''} {parsed['x']['shape']}".strip())
        y = choose_unique(my, f"{parsed['y']['color'] or ''} {parsed['y']['shape']}".strip())
        plan = plan_put(world, x, y)
        execute_plan(world, plan)
        return f"OK. Put {x} on {y}. Plan: {plan}"

    else:
        raise ValueError("Unsupported intent.")


# ----------------------------------------
# 6) Visualization
# ----------------------------------------

def visualize_world(world: World) -> Image.Image:
    """Create a visual representation of the blocks world."""
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.set_xlim(-1, 10)
    ax.set_ylim(-0.5, 5)
    ax.set_aspect('equal')
    ax.axis('off')
    
    # Build stacks for each object on the table
    stacks = {}
    x_pos = 0
    
    # Find all objects on the table
    on_table = [name for name, support in world.on.items() if support is None and name != world.holding]
    
    for table_obj in sorted(on_table):
        stack = [table_obj]
        # Build stack upward
        current = table_obj
        while True:
            top = world.top_of(current)
            if top is None:
                break
            stack.append(top)
            current = top
        stacks[table_obj] = stack
        
    # Draw each stack
    x_pos = 0.5
    for base_obj, stack in stacks.items():
        y_pos = 0
        for obj_name in stack:
            obj = world.objects[obj_name]
            
            if obj.shape == "pyramid":
                # Draw triangle
                points = [
                    [x_pos, y_pos],
                    [x_pos + 0.5, y_pos],
                    [x_pos + 0.25, y_pos + 0.6]
                ]
                triangle = patches.Polygon(points, closed=True, 
                                         edgecolor='black', facecolor=obj.color, 
                                         linewidth=2)
                ax.add_patch(triangle)
                ax.text(x_pos + 0.25, y_pos + 0.2, obj_name, 
                       ha='center', va='center', fontsize=10, fontweight='bold')
                y_pos += 0.7
            else:  # cube or block
                rect = patches.Rectangle((x_pos, y_pos), 0.5, 0.5,
                                        edgecolor='black', facecolor=obj.color,
                                        linewidth=2)
                ax.add_patch(rect)
                ax.text(x_pos + 0.25, y_pos + 0.25, obj_name,
                       ha='center', va='center', fontsize=10, fontweight='bold')
                y_pos += 0.55
        
        x_pos += 1.2
    
    # Draw held object
    if world.holding:
        obj = world.objects[world.holding]
        hold_x, hold_y = 8, 3.5
        
        if obj.shape == "pyramid":
            points = [
                [hold_x, hold_y],
                [hold_x + 0.5, hold_y],
                [hold_x + 0.25, hold_y + 0.6]
            ]
            triangle = patches.Polygon(points, closed=True,
                                     edgecolor='black', facecolor=obj.color,
                                     linewidth=2, linestyle='--')
            ax.add_patch(triangle)
        else:
            rect = patches.Rectangle((hold_x, hold_y), 0.5, 0.5,
                                    edgecolor='black', facecolor=obj.color,
                                    linewidth=2, linestyle='--')
            ax.add_patch(rect)
        
        ax.text(hold_x + 0.25, hold_y - 0.3, f"HOLDING: {world.holding}",
               ha='center', fontsize=10, fontweight='bold')
    
    # Draw table line
    ax.axhline(y=0, color='brown', linewidth=4, linestyle='-', alpha=0.7)
    ax.text(5, -0.3, "TABLE", ha='center', fontsize=12, fontweight='bold', color='brown')
    
    plt.tight_layout()
    
    # Convert to image
    buf = BytesIO()
    plt.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    buf.seek(0)
    img = Image.open(buf)
    plt.close(fig)
    
    return img


# ----------------------------------------
# 7) Gradio Interface
# ----------------------------------------

# Initialize world state
def create_initial_world():
    return World(
        objects={
            "b1": Obj("b1", "red", "cube"),
            "b2": Obj("b2", "yellow", "cube"),
            "p1": Obj("p1", "blue", "pyramid"),
            "c1": Obj("c1", "green", "cube"),
        },
        on={
            "p1": "b1",
            "b1": None,
            "b2": None,
            "c1": None,
        },
    )

# Global world state
world_state = create_initial_world()

def process_command(command: str, history: List[Tuple[str, str]]) -> Tuple[List[Tuple[str, str]], Image.Image, str]:
    """Process a command and update the world."""
    global world_state
    
    if not command.strip():
        return history, visualize_world(world_state), world_state.describe()
    
    try:
        response = interpret_and_act(world_state, command)
        history.append((command, response))
    except (ValueError, RuntimeError) as e:
        response = f"❌ {str(e)}"
        history.append((command, response))
    
    return history, visualize_world(world_state), world_state.describe()

def reset_world():
    """Reset the world to initial state."""
    global world_state
    world_state = create_initial_world()
    return [], visualize_world(world_state), world_state.describe()

# Build Gradio interface
with gr.Blocks(title="SHRDLU Blocks World", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🧱 SHRDLU Blocks World Simulator
    
    A modern implementation of Terry Winograd's classic SHRDLU natural language understanding system.
    
    **Try commands like:**
    - "pick up the blue pyramid"
    - "put the blue pyramid on the red cube"
    - "put the green cube on the blue pyramid"
    """)
    
    with gr.Row():
        with gr.Column(scale=1):
            chatbot = gr.Chatbot(
                label="Conversation History",
                height=400,
                show_label=True,
                bubble_full_width=False
            )
            
            with gr.Row():
                command_input = gr.Textbox(
                    label="Enter Command",
                    placeholder="e.g., put the blue pyramid on the red cube",
                    scale=4
                )
                submit_btn = gr.Button("Execute", variant="primary", scale=1)
            
            with gr.Row():
                reset_btn = gr.Button("🔄 Reset World", variant="secondary")
            
            world_text = gr.Textbox(
                label="World State (Text)",
                lines=6,
                interactive=False
            )
        
        with gr.Column(scale=1):
            world_viz = gr.Image(
                label="World Visualization",
                type="pil",
                height=500
            )
    
    gr.Markdown("""
    ### 📝 Notes:
    - Objects are identified by **color** and **shape**
    - The system will ask for clarification if the description is ambiguous
    - The planner automatically moves blocking objects out of the way
    - Dashed outline indicates an object being held
    """)
    
    # Event handlers
    submit_btn.click(
        fn=process_command,
        inputs=[command_input, chatbot],
        outputs=[chatbot, world_viz, world_text]
    ).then(
        lambda: "",
        outputs=[command_input]
    )
    
    command_input.submit(
        fn=process_command,
        inputs=[command_input, chatbot],
        outputs=[chatbot, world_viz, world_text]
    ).then(
        lambda: "",
        outputs=[command_input]
    )
    
    reset_btn.click(
        fn=reset_world,
        outputs=[chatbot, world_viz, world_text]
    )
    
    # Load initial state
    demo.load(
        fn=lambda: ([], visualize_world(world_state), world_state.describe()),
        outputs=[chatbot, world_viz, world_text]
    )

if __name__ == "__main__":
    demo.launch(share=False, server_name="0.0.0.0")

/var/folders/jl/y_4hb2553llg1s2py_vpnp4w0000gn/T/ipykernel_72375/4101270465.py:364: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/var/folders/jl/y_4hb2553llg1s2py_vpnp4w0000gn/T/ipykernel_72375/4101270465.py:364: DeprecationWarning: The 'bubble_full_width' parameter is deprecated and will be removed in a future version. This parameter no longer has any effect.
  chatbot = gr.Chatbot(
ERROR:    [Errno 48] error while attempting to bind on address ('0.0.0.0', 7860): address already in use


* Running on local URL:  http://0.0.0.0:7861
* To create a public link, set `share=True` in `launch()`.
